In [1]:
pip install web3 cryptography requests

   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.5 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.5 MB 1.5 MB/s eta 0:00:03
   -------- ------------------------------- 0.8/3.5 MB 1.3 MB/s eta 0:00:03
   ----------- ---------------------------- 1.0/3.5 MB 1.1 MB/s eta 0:00:03
   -------------- ------------------------- 1.3/3.5 MB 1.2 MB/s eta 0:00:02
   ----------------- ---------------------- 1.6/3.5 MB 1.2 MB/s eta 0:00:02
   -------------------- ------------------- 1.8/3.5 MB 1.1 MB/s eta 0:00:02
   ----------------------- ---------------- 2.1/3.5 MB 1.2 MB/s eta 0:00:02
   -------------------------- ------------- 2.4/3.5 MB 1.2 MB/s eta 0:00:01
   ----------------------------- ---------- 2.6/3.5 MB 1.2 MB/s eta 0:00:01
   ----------------------------- ---------- 2.6/3.5 MB 1.2 MB/s eta 0:00:01
   ------------------------------

In [ ]:
import json
import time
from web3 import Web3

# ==========================================
# 1. CONFIGURATION 
# ==========================================

# Your Alchemy URL
INFURA_URL = "https://eth-sepolia.g.alchemy.com/v2/yQnCLGGsGJKH8rISuFzXg"

# Your Private Key
PRIVATE_KEY = "dc46923fe29ae8b022394667e0b32fe13bc7d12713127465effd9d3eaf745380"

# Connect FIRST so we can use the helper tool
web3 = Web3(Web3.HTTPProvider(INFURA_URL))

if web3.is_connected():
    print(f"✅ Connected to Sepolia! Block: {web3.eth.block_number}")
else:
    print("❌ Connection Failed. Check your internet.")
    exit()

# --- THE FIX IS HERE ---
# We use 'to_checksum_address' to fix the lowercase error automatically
SENDER_ADDRESS = web3.to_checksum_address("0xe7987961c7de22ca7242dac44474c0f553912e65")
CONTRACT_ADDRESS = web3.to_checksum_address("0x79DE06d0E587662EEF56eb187c8007Dc253c0e80")
COLLEAGUE_ADDRESS = web3.to_checksum_address("0x71C7656EC7ab88b098defB751B7401B5f6d8976F")

# ==========================================
# 2. CONTRACT SETUP
# ==========================================
# The ABI for your 'SecureCloud' contract
CONTRACT_ABI = json.loads('''
[
    {
        "anonymous": false,
        "inputs": [
            {"indexed": true, "internalType": "uint256", "name": "fileId", "type": "uint256"},
            {"indexed": true, "internalType": "address", "name": "owner", "type": "address"},
            {"indexed": true, "internalType": "address", "name": "user", "type": "address"}
        ],
        "name": "AccessGranted",
        "type": "event"
    },
    {
        "inputs": [
            {"internalType": "uint256", "name": "_fileId", "type": "uint256"},
            {"internalType": "address", "name": "_user", "type": "address"}
        ],
        "name": "grantAccess",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function"
    },
    {
        "inputs": [
            {"internalType": "uint256", "name": "_fileId", "type": "uint256"},
            {"internalType": "string", "name": "_ipfsCID", "type": "string"},
            {"internalType": "string", "name": "_fileName", "type": "string"},
            {"internalType": "uint256", "name": "_fileSize", "type": "uint256"}
        ],
        "name": "uploadFile",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function"
    },
    {
        "inputs": [
            {"internalType": "uint256", "name": "_fileId", "type": "uint256"},
            {"internalType": "address", "name": "_user", "type": "address"}
        ],
        "name": "verifyAccess",
        "outputs": [{"internalType": "bool", "name": "", "type": "bool"}],
        "stateMutability": "view",
        "type": "function"
    }
]
''')

contract = web3.eth.contract(address=CONTRACT_ADDRESS, abi=CONTRACT_ABI)

# ==========================================
# 3. HELPER FUNCTION: SIGN & SEND
# ==========================================
def send_transaction(function_call, description):
    print(f"\n⏳ {description}...")
    
    # Get the latest transaction count (nonce)
    nonce = web3.eth.get_transaction_count(SENDER_ADDRESS)
    
    # Build the transaction
    # CHANGE: Lowered gas from 2,000,000 to 500,000 to fit your balance
    tx_data = function_call.build_transaction({
        'chainId': 11155111,  # Sepolia Chain ID
        'gas': 500000,        # <--- LOWERED THIS (Was 2000000)
        'maxFeePerGas': web3.to_wei('50', 'gwei'),
        'maxPriorityFeePerGas': web3.to_wei('2', 'gwei'),
        'nonce': nonce,
        'from': SENDER_ADDRESS
    })
    
    # Sign it with your private key
    signed_tx = web3.eth.account.sign_transaction(tx_data, PRIVATE_KEY)
    
    # Send it to the network
    tx_hash = web3.eth.send_raw_transaction(signed_tx.raw_transaction)
    
    # Wait for confirmation
    receipt = web3.eth.wait_for_transaction_receipt(tx_hash)
    print(f"✅ Confirmed! Gas Used: {receipt.gasUsed}")
    print(f"🔗 Tx Hash: https://sepolia.etherscan.io/tx/{receipt.transactionHash.hex()}")
    return receipt

# ==========================================
# 4. MAIN EXECUTION SCENARIO
# ==========================================
if __name__ == "__main__":
    
    # --- RANDOM FILE ID GENERATION ---
    # This ensures every test run is unique and avoids "ID Collision" errors
    file_id = int(time.time()) 
    
    filename = "Final_Dissertation_Proof.pdf"
    ipfs_hash = "QmXoypizjW3WknFiJnKLwHCnL72vedxjQkDDP1mXWo6uco" 
    file_size = 5120

    try:
        print(f"--- 🚀 Starting Test for File ID: {file_id} ---")

        # STEP 1: UPLOAD
        upload_call = contract.functions.uploadFile(file_id, ipfs_hash, filename, file_size)
        send_transaction(upload_call, f"Uploading Metadata")

        # STEP 2: VERIFY ACCESS (Owner)
        print("\n🔍 Verifying Owner Access...")
        has_access = contract.functions.verifyAccess(file_id, SENDER_ADDRESS).call()
        print(f"👉 Access Status: {'GRANTED' if has_access else 'DENIED'}")

        # STEP 3: GRANT ACCESS (To a dummy colleague)
        grant_call = contract.functions.grantAccess(file_id, COLLEAGUE_ADDRESS)
        send_transaction(grant_call, f"Granting Access to Colleague")
        
        # STEP 4: VERIFY COLLEAGUE ACCESS
        print("\n🔍 Verifying Colleague Access...")
        colleague_access = contract.functions.verifyAccess(file_id, COLLEAGUE_ADDRESS).call()
        print(f"👉 Colleague Access Status: {'GRANTED' if colleague_access else 'DENIED'}")

    except Exception as e:
        print(f"\n❌ ERROR: {e}")

✅ Connected to Sepolia! Block: 9813847
--- 🚀 Starting Test for File ID: 1765416474 ---

⏳ Uploading Metadata...
✅ Confirmed! Gas Used: 206421
🔗 Tx Hash: https://sepolia.etherscan.io/tx/0efed84f1c2069fb2b18722512c0718e7a3590512220d68aee1d7cb693871577

🔍 Verifying Owner Access...
👉 Access Status: GRANTED

⏳ Granting Access to Colleague...
✅ Confirmed! Gas Used: 48712
🔗 Tx Hash: https://sepolia.etherscan.io/tx/998e9292950493337ba5bba716fe485ca73e2d6c1dd1b4f3cdc0184a2a3bb0e9

🔍 Verifying Colleague Access...
👉 Colleague Access Status: GRANTED
